# Part 3 — Full-Image Transfer Learning (Swin-Tiny)

**Backbone:** Swin Transformer Tiny — frozen ImageNet weights from `torchvision`  
**Why Swin-Tiny?** Smallest Swin variant (~28M params total, only ~600K trainable). Runs one epoch in ~2 min on Colab T4. No extra pip installs beyond scikit-learn.

### Prerequisites — files needed in Google Drive:
| File | Source |
|------|--------|
| `train_ende.zip` | MSCTD dataset |
| `dev.zip` | MSCTD dataset |
| `test.zip` | MSCTD dataset |
| `sentiment_train.txt` | MSCTD labels |
| `sentiment_dev.txt` | MSCTD labels |
| `sentiment_test.txt` | MSCTD labels |

**Run cells in order from top to bottom.**

In [ ]:
# CELL 1 — Mount Drive + minimal install
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-learn
!pip install --upgrade --force-reinstall "Pillow<10" -q
print('Done.')

In [ ]:
# CELL 2 — Unzip dataset images
import os, glob

# ── EDIT THESE if your files are in a different Drive folder ──────────────────
DRIVE_ROOT  = '/content/drive/MyDrive'       # weights + output plots saved here
DRIVE_AML   = '/content/drive/MyDrive/AML'   # sentiment_*.txt files + dataset zips
# ─────────────────────────────────────────────────────────────────────────────

def find_file(name):
    for pattern in [f'{DRIVE_ROOT}/{name}', f'{DRIVE_AML}/{name}',
                    f'{DRIVE_ROOT}/**/{name}']:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    return None

DATA_DIR = '/content/dataset'
os.makedirs(DATA_DIR, exist_ok=True)

for zip_name, target in [('train_ende.zip', f'{DATA_DIR}/train_ende'),
                          ('dev.zip',        f'{DATA_DIR}/dev'),
                          ('test.zip',       f'{DATA_DIR}/test')]:
    if not os.path.exists(target):
        p = find_file(zip_name)
        if p:
            print(f'Unzipping {zip_name} ...')
            os.system(f'unzip -q "{p}" -d {DATA_DIR}/')
        else:
            raise FileNotFoundError(f'{zip_name} not found on Drive. Check DRIVE_AML path.')
    else:
        print(f'{os.path.basename(target)}: already on disk.')

print('Dataset folders:', os.listdir(DATA_DIR))

In [ ]:
# CELL 3 — Imports & config
import os, random, copy, warnings
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import Swin_T_Weights
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import (f1_score, confusion_matrix,
                              ConfusionMatrixDisplay, classification_report)
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

LABEL_MAP   = {0: 'Neutral', 1: 'Negative', 2: 'Positive'}
NUM_CLASSES = 3

# Label file paths
LABEL_TRAIN = f'{DRIVE_AML}/sentiment_train.txt'
LABEL_DEV   = f'{DRIVE_AML}/sentiment_dev.txt'
LABEL_TEST  = f'{DRIVE_AML}/sentiment_test.txt'

# ── Speed-optimised hyper-params for Colab T4 ────────────────────────────────
IMG_SIZE    = 224      # Swin-Tiny native resolution
BATCH_SIZE  = 64       # fits in T4 16 GB with frozen backbone
NUM_EPOCHS  = 15       # ~2 min/epoch → ≤30 min total
LR          = 3e-4
WEIGHT_DECAY= 1e-4
PATIENCE    = 5        # early stop

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
print('Config OK.')

In [ ]:
# CELL 4 — Auto-detect image sub-folder & build Dataset

def find_image_dir(base):
    """Return the subfolder that actually contains image files."""
    for d in [base, os.path.join(base,'images'), os.path.join(base,'image')]:
        if os.path.isdir(d) and any(
            f.lower().endswith(('.jpg','.jpeg','.png')) for f in os.listdir(d)
        ):
            return d
    for root,_,files in os.walk(base):
        if any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in files):
            return root
    raise FileNotFoundError(f'No images found under {base}')


class FullImageDataset(Dataset):
    def __init__(self, img_dir, label_file, transform):
        self.img_dir   = img_dir
        self.transform = transform
        all_files = [f for f in os.listdir(img_dir)
                     if f.lower().endswith(('.jpg','.jpeg','.png'))]
        # Sort numerically by filename stem
        self.images = sorted(
            all_files,
            key=lambda x: int(os.path.splitext(x)[0])
                          if os.path.splitext(x)[0].isdigit() else x
        )
        with open(label_file) as f:
            self.labels = [int(l.strip()) for l in f if l.strip()]
        assert len(self.images) == len(self.labels), \
            f'Image/label count mismatch: {len(self.images)} vs {len(self.labels)}'

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        try:
            img = Image.open(os.path.join(self.img_dir, self.images[i])).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        return self.transform(img), self.labels[i]


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

IMG_TRAIN_DIR = find_image_dir('/content/dataset/train_ende')
IMG_DEV_DIR   = find_image_dir('/content/dataset/dev')
IMG_TEST_DIR  = find_image_dir('/content/dataset/test')

train_ds = FullImageDataset(IMG_TRAIN_DIR, LABEL_TRAIN, train_tf)
dev_ds   = FullImageDataset(IMG_DEV_DIR,   LABEL_DEV,   eval_tf)
test_ds  = FullImageDataset(IMG_TEST_DIR,  LABEL_TEST,  eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}  Dev: {len(dev_ds)}  Test: {len(test_ds)}')
print(f'Image dirs OK — Train: {IMG_TRAIN_DIR}')

In [ ]:
# CELL 5 — Build model: frozen Swin-Tiny backbone + custom head
#
# Swin-Tiny output: 768-d vector from the final norm layer.
# Custom head: BN → FC(768→256) → GELU → Dropout(0.4) → FC(256→3)
# Only ~600K parameters are trainable — very fast to train.

class SwinSentimentClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # ── Frozen Swin-Tiny backbone ─────────────────────────────────────────
        swin = models.swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
        self.feature_dim = swin.head.in_features   # 768
        swin.head = nn.Identity()                  # remove classification head
        self.backbone = swin

        for p in self.backbone.parameters():       # FREEZE
            p.requires_grad = False

        # ── Trainable classification head ─────────────────────────────────────
        self.head = nn.Sequential(
            nn.BatchNorm1d(self.feature_dim),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        with torch.no_grad():
            feats = self.backbone(x)   # (B, 768)
        return self.head(feats)         # (B, 3)


model = SwinSentimentClassifier(NUM_CLASSES).to(device)

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Backbone       : Swin-Tiny (ImageNet V1)  — FROZEN')
print(f'Feature dim    : {model.feature_dim}')
print(f'Total params   : {total:,}')
print(f'Trainable      : {trainable:,}  ← only the head trains')
print(f'Frozen         : {total - trainable:,}')

In [ ]:
# CELL 6 — Loss, optimiser, scheduler

label_counts  = Counter(train_ds.labels)
class_weights = torch.tensor(
    [len(train_ds) / (NUM_CLASSES * label_counts[c]) for c in range(NUM_CLASSES)],
    dtype=torch.float32
).to(device)
print('Class weights:', {LABEL_MAP[i]: f'{class_weights[i]:.3f}' for i in range(NUM_CLASSES)})

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print('Optimiser: AdamW | Scheduler: CosineAnnealingLR')

In [ ]:
# CELL 7 — Training loop with early stopping

BEST_MODEL_PATH = f'{DRIVE_ROOT}/best_fullimage_model_part3.pth'

def run_epoch(model, loader, criterion, optimizer=None, desc=''):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, desc=desc, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if is_train: optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
            preds = logits.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds .extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return total_loss / total, correct / total, all_preds, all_labels


history = {'train_loss':[], 'train_acc':[], 'dev_loss':[], 'dev_acc':[]}
best_dev_acc = 0.0
no_improve   = 0

print(f'Training Swin-Tiny head for up to {NUM_EPOCHS} epochs (patience={PATIENCE})\n')
print(f'{"Ep":>3} {"T-Loss":>8} {"T-Acc":>7} {"D-Loss":>8} {"D-Acc":>7} {"LR":>9}')
print('-' * 50)

for ep in range(1, NUM_EPOCHS + 1):
    tl, ta, _, _  = run_epoch(model, train_loader, criterion, optimizer, f'Ep{ep} train')
    dl, da, _, _  = run_epoch(model, dev_loader,   criterion, desc=f'Ep{ep} dev')
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['dev_loss'  ].append(dl); history['dev_acc'  ].append(da)

    flag = ''
    if da > best_dev_acc:
        best_dev_acc = da
        torch.save(copy.deepcopy(model.state_dict()), BEST_MODEL_PATH)
        flag = ' ★'
        no_improve = 0
    else:
        no_improve += 1

    print(f'{ep:>3} {tl:>8.4f} {ta:>7.4f} {dl:>8.4f} {da:>7.4f} {lr:>9.2e}{flag}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stop — no improvement for {PATIENCE} epochs.')
        break

print(f'\nBest dev accuracy : {best_dev_acc:.4f}')
print(f'Model saved       : {BEST_MODEL_PATH}')

In [ ]:
# CELL 8 — Training curves

ep_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(ep_range, history['train_loss'], 'o-', label='Train')
ax1.plot(ep_range, history['dev_loss'],   's-', label='Dev')
ax1.set(xlabel='Epoch', ylabel='Loss', title='Part 3 — Loss'); ax1.legend(); ax1.grid(0.3)

ax2.plot(ep_range, history['train_acc'], 'o-', label='Train')
ax2.plot(ep_range, history['dev_acc'],   's-', label='Dev')
ax2.axhline(best_dev_acc, color='green', ls='--', lw=1, label=f'Best dev {best_dev_acc:.4f}')
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Part 3 — Accuracy', ylim=(0,1))
ax2.legend(); ax2.grid(0.3)

plt.suptitle('Part 3 — Swin-Tiny Frozen Backbone: Training Curves', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/training_curves_part3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to Drive.')

In [ ]:
# CELL 9 — Load best checkpoint and evaluate on test set

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
print(f'Loaded: {BEST_MODEL_PATH}\n')

_, test_acc, test_preds, test_labels = run_epoch(model, test_loader, criterion, desc='Test')
macro_f1    = f1_score(test_labels, test_preds, average='macro')
weighted_f1 = f1_score(test_labels, test_preds, average='weighted')

print('=' * 48)
print('PART 3 — TEST RESULTS  (Swin-Tiny frozen)')
print('=' * 48)
print(f'  Accuracy     : {test_acc:.4f}')
print(f'  Macro F1     : {macro_f1:.4f}')
print(f'  Weighted F1  : {weighted_f1:.4f}')

In [ ]:
# CELL 10 — Per-class report

print(classification_report(
    test_labels, test_preds,
    target_names=[LABEL_MAP[i] for i in range(NUM_CLASSES)]
))

In [ ]:
# CELL 11 — Confusion matrix

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=[LABEL_MAP[i] for i in range(NUM_CLASSES)]).plot(
    ax=ax, colorbar=True, cmap='Blues'
)
ax.set_title(
    f'Confusion Matrix — Part 3 (Swin-Tiny frozen)\n'
    f'Accuracy: {test_acc:.3f}  |  Macro F1: {macro_f1:.3f}', fontsize=10
)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/confusion_matrix_part3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to Drive.')

In [ ]:
# CELL 12 — Visualise sample test predictions

inv_norm = transforms.Normalize(
    mean=[-m/s for m, s in zip(MEAN, STD)], std=[1/s for s in STD]
)

model.eval()
sample_imgs, sample_true, sample_pred = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(device)).argmax(1).cpu()
        sample_imgs .extend(imgs[:16])
        sample_true .extend(labels[:16].tolist())
        sample_pred .extend(preds[:16].tolist())
        if len(sample_imgs) >= 16: break

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for ax, img, t, p in zip(axes.flatten(), sample_imgs[:16], sample_true, sample_pred):
    ax.imshow(inv_norm(img).clamp(0,1).permute(1,2,0).numpy())
    ok = t == p
    ax.set_title(f'T:{LABEL_MAP[t]}\nP:{LABEL_MAP[p]} {"✓" if ok else "✗"}',
                 color='green' if ok else 'red', fontsize=7)
    ax.axis('off')
plt.suptitle('Part 3 — Sample Test Predictions (Swin-Tiny)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/sample_predictions_part3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to Drive.')

In [ ]:
# CELL 13 — Save softmax probs for Part 4 fusion

model.eval()

def extract_probs(loader, desc):
    probs_list, label_list = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc):
            p = torch.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
            probs_list.extend(p.tolist())
            label_list.extend(labels.tolist())
    return np.array(probs_list), np.array(label_list)

test_probs, test_lbls = extract_probs(test_loader, 'Test probs')
dev_probs,  dev_lbls  = extract_probs(dev_loader,  'Dev probs')

np.save(f'{DRIVE_ROOT}/part3_test_probs.npy',  test_probs)
np.save(f'{DRIVE_ROOT}/part3_test_labels.npy', test_lbls)
np.save(f'{DRIVE_ROOT}/part3_dev_probs.npy',   dev_probs)
np.save(f'{DRIVE_ROOT}/part3_dev_labels.npy',  dev_lbls)

print(f'Saved  test probs : {test_probs.shape}')
print(f'Saved  dev probs  : {dev_probs.shape}')
print('These .npy files are needed by Part 4 fusion.')

In [ ]:
# CELL 14 — Final summary

print('=' * 58)
print('PART 3 — SWIN-TINY FROZEN BACKBONE — FINAL SUMMARY')
print('=' * 58)
print(f'  Backbone       : Swin-Tiny (ImageNet V1) — FROZEN')
print(f'  Feature dim    : {model.feature_dim}')
print(f'  Head           : BN → FC(768→256) → GELU → Drop(0.4) → FC(256→3)')
print(f'  Trainable      : {trainable:,} params')
print(f'  Epochs trained : {len(history["train_loss"])}')
print(f'  Best dev acc   : {best_dev_acc:.4f}')
print()
print(f'  Test Accuracy  : {test_acc:.4f}')
print(f'  Macro F1       : {macro_f1:.4f}')
print(f'  Weighted F1    : {weighted_f1:.4f}')
print()
print('  Files saved to Drive:')
print('    best_fullimage_model_part3.pth   ← Part 4 fusion')
print('    part3_test_probs.npy             ← Part 4 fusion')
print('    part3_dev_probs.npy              ← Part 4 fusion')
print('    training_curves_part3.png')
print('    confusion_matrix_part3.png')
print('    sample_predictions_part3.png')